# The Text Editor Tool

Claude ships one **built-in** tool you don't define from scratch: the **text editor**. Claude already knows its schema — you just include a small stub and **implement the file operations yourself**. It gives Claude view / create / edit powers over your files, effectively turning it into a hands-on code editor.

**Commands (current version):** `view`, `str_replace`, `create`, `insert`.

**The split of responsibilities:**
- **Claude provides** the tool schema knowledge (you send only a tiny type/name stub).
- **You provide** the actual functions that read directories, view files, replace text, create files, insert lines.

> **Adaptations from the course:**
> - **Schema version.** The lesson maps only old models (`claude-3-7-sonnet` → `text_editor_20250124`, etc.). For Claude 4.x models the current stub is **`text_editor_20250728`** / name **`str_replace_based_edit_tool`** (confirmed against the docs). `undo_edit` was **removed** in `text_editor_20250429`+, so Claude won't call it (the method stays as harmless dead code; backups still run on edits).
> - **Bug fix.** The reference's `run_tool` checks `tool_name == "str_replace_editor"` — the *old* name — so it'd reject every call. Fixed to `str_replace_based_edit_tool`.
> - **Model → `claude-haiku-4-5-20251001`** (a Claude 4.x model; the reference's `claude-sonnet-4-5` + hard-coded `temperature=1.0` would 400 on flagships).
> - **Sandboxed** to a `text-editor-sandbox/` folder so Claude edits there, not the whole repo (the tool's `_validate_path` blocks escaping `base_dir`).

## Setup — client, helpers

In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

import os
from anthropic import Anthropic
from anthropic.types import Message

client = Anthropic()
model = "claude-haiku-4-5-20251001"


def add_user_message(messages, message):
    messages.append({
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    })


def add_assistant_message(messages, message):
    messages.append({
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    })


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }
    if tools:
        params["tools"] = tools
    if system:
        params["system"] = system
    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])


print(f"Ready. Model: {model}")

Ready. Model: claude-haiku-4-5-20251001


## The `TextEditorTool` implementation

This is *your* half of the tool: a sandboxed file editor with view / str_replace / create / insert (+ an undo backed by timestamped backups, though the current tool version won't trigger it). `_validate_path` keeps every operation inside `base_dir`.

In [2]:
import shutil
from typing import Optional, List


class TextEditorTool:
    def __init__(self, base_dir: str = "", backup_dir: str = ""):
        self.base_dir = base_dir or os.getcwd()
        self.backup_dir = backup_dir or os.path.join(self.base_dir, ".backups")
        os.makedirs(self.backup_dir, exist_ok=True)

    def _validate_path(self, file_path: str) -> str:
        abs_path = os.path.normpath(os.path.join(self.base_dir, file_path))
        if not abs_path.startswith(self.base_dir):
            raise ValueError(
                f"Access denied: Path '{file_path}' is outside the allowed directory"
            )
        return abs_path

    def _backup_file(self, file_path: str) -> str:
        if not os.path.exists(file_path):
            return ""
        file_name = os.path.basename(file_path)
        backup_path = os.path.join(
            self.backup_dir, f"{file_name}.{os.path.getmtime(file_path):.0f}"
        )
        shutil.copy2(file_path, backup_path)
        return backup_path

    def _restore_backup(self, file_path: str) -> str:
        file_name = os.path.basename(file_path)
        backups = [
            f for f in os.listdir(self.backup_dir) if f.startswith(file_name + ".")
        ]
        if not backups:
            raise FileNotFoundError(f"No backups found for {file_path}")

        latest_backup = sorted(backups, reverse=True)[0]
        backup_path = os.path.join(self.backup_dir, latest_backup)

        shutil.copy2(backup_path, file_path)
        return f"Successfully restored {file_path} from backup"

    def _count_matches(self, content: str, old_str: str) -> int:
        return content.count(old_str)

    def view(self, file_path: str, view_range: Optional[List[int]] = None) -> str:
        try:
            abs_path = self._validate_path(file_path)

            if os.path.isdir(abs_path):
                try:
                    return "\n".join(os.listdir(abs_path))
                except PermissionError:
                    raise PermissionError(
                        "Permission denied. Cannot list directory contents."
                    )

            if not os.path.exists(abs_path):
                raise FileNotFoundError("File not found")

            with open(abs_path, "r", encoding="utf-8") as f:
                content = f.read()

            if view_range:
                start, end = view_range
                lines = content.split("\n")

                if end == -1:
                    end = len(lines)

                selected_lines = lines[start - 1 : end]

                result = []
                for i, line in enumerate(selected_lines, start):
                    result.append(f"{i}: {line}")

                return "\n".join(result)
            else:
                lines = content.split("\n")
                result = []
                for i, line in enumerate(lines, 1):
                    result.append(f"{i}: {line}")

                return "\n".join(result)

        except UnicodeDecodeError:
            raise UnicodeDecodeError(
                "utf-8", b"", 0, 1,
                "File contains non-text content and cannot be displayed.",
            )
        except ValueError as e:
            raise ValueError(str(e))
        except PermissionError:
            raise PermissionError("Permission denied. Cannot access file.")
        except Exception as e:
            raise type(e)(str(e))

    def str_replace(self, file_path: str, old_str: str, new_str: str) -> str:
        try:
            abs_path = self._validate_path(file_path)

            if not os.path.exists(abs_path):
                raise FileNotFoundError("File not found")

            with open(abs_path, "r", encoding="utf-8") as f:
                content = f.read()

            match_count = self._count_matches(content, old_str)

            if match_count == 0:
                raise ValueError(
                    "No match found for replacement. Please check your text and try again."
                )
            elif match_count > 1:
                raise ValueError(
                    f"Found {match_count} matches for replacement text. Please provide more context to make a unique match."
                )

            self._backup_file(abs_path)

            new_content = content.replace(old_str, new_str)

            with open(abs_path, "w", encoding="utf-8") as f:
                f.write(new_content)

            return "Successfully replaced text at exactly one location."

        except ValueError as e:
            raise ValueError(str(e))
        except PermissionError:
            raise PermissionError("Permission denied. Cannot modify file.")
        except Exception as e:
            raise type(e)(str(e))

    def create(self, file_path: str, file_text: str) -> str:
        try:
            abs_path = self._validate_path(file_path)

            if os.path.exists(abs_path):
                raise FileExistsError(
                    "File already exists. Use str_replace to modify it."
                )

            os.makedirs(os.path.dirname(abs_path), exist_ok=True)

            with open(abs_path, "w", encoding="utf-8") as f:
                f.write(file_text)

            return f"Successfully created {file_path}"

        except ValueError as e:
            raise ValueError(str(e))
        except PermissionError:
            raise PermissionError("Permission denied. Cannot create file.")
        except Exception as e:
            raise type(e)(str(e))

    def insert(self, file_path: str, insert_line: int, new_str: str) -> str:
        try:
            abs_path = self._validate_path(file_path)

            if not os.path.exists(abs_path):
                raise FileNotFoundError("File not found")

            self._backup_file(abs_path)

            with open(abs_path, "r", encoding="utf-8") as f:
                lines = f.readlines()

            if lines and not lines[-1].endswith("\n"):
                new_str = "\n" + new_str

            if insert_line == 0:
                lines.insert(0, new_str + "\n")
            elif insert_line > 0 and insert_line <= len(lines):
                lines.insert(insert_line, new_str + "\n")
            else:
                raise IndexError(
                    f"Line number {insert_line} is out of range. File has {len(lines)} lines."
                )

            with open(abs_path, "w", encoding="utf-8") as f:
                f.writelines(lines)

            return f"Successfully inserted text after line {insert_line}"

        except ValueError as e:
            raise ValueError(str(e))
        except PermissionError:
            raise PermissionError("Permission denied. Cannot modify file.")
        except Exception as e:
            raise type(e)(str(e))

    def undo_edit(self, file_path: str) -> str:
        # Note: text_editor_20250728 removed the undo_edit command, so Claude won't call this.
        try:
            abs_path = self._validate_path(file_path)

            if not os.path.exists(abs_path):
                raise FileNotFoundError("File not found")

            return self._restore_backup(abs_path)

        except ValueError as e:
            raise ValueError(str(e))
        except FileNotFoundError:
            raise FileNotFoundError("No previous edits to undo")
        except PermissionError:
            raise PermissionError("Permission denied. Cannot restore file.")
        except Exception as e:
            raise type(e)(str(e))

## Sandbox + a sample file to work on

Point the tool at a dedicated folder and seed a small `main.py` so the "summarize this file" demo has something to read.

In [3]:
SECTION = os.path.join(os.path.dirname(find_dotenv()), "building-with-the-claude-api", "04-tool-use")
SANDBOX = os.path.join(SECTION, "text-editor-sandbox")
os.makedirs(SANDBOX, exist_ok=True)

SAMPLE_MAIN = '''def greet(name):
    return f"Hello, {name}!"


def add(a, b):
    return a + b


if __name__ == "__main__":
    print(greet("world"))
    print(add(2, 3))
'''

with open(os.path.join(SANDBOX, "main.py"), "w", encoding="utf-8") as f:
    f.write(SAMPLE_MAIN)

text_editor_tool = TextEditorTool(base_dir=SANDBOX)
print("Sandbox:", SANDBOX)

Sandbox: c:\Development\anthropic-cert\building-with-the-claude-api\04-tool-use\text-editor-sandbox


## Route tool calls + the schema stub

`run_tool` dispatches each editor command to the class. **The fix:** match the real tool name `str_replace_based_edit_tool` (not the old `str_replace_editor`). The schema is just a two-line stub — Claude expands it internally.

In [4]:
import json


def run_tool(tool_name, tool_input):
    if tool_name == "str_replace_based_edit_tool":
        command = tool_input["command"]
        if command == "view":
            return text_editor_tool.view(tool_input["path"], tool_input.get("view_range"))
        elif command == "str_replace":
            return text_editor_tool.str_replace(
                tool_input["path"], tool_input["old_str"], tool_input["new_str"]
            )
        elif command == "create":
            return text_editor_tool.create(tool_input["path"], tool_input["file_text"])
        elif command == "insert":
            return text_editor_tool.insert(
                tool_input["path"], tool_input["insert_line"], tool_input["new_str"]
            )
        elif command == "undo_edit":
            return text_editor_tool.undo_edit(tool_input["path"])
        else:
            raise Exception(f"Unknown text editor command: {command}")
    else:
        raise Exception(f"Unknown tool name: {tool_name}")


def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []
    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_blocks.append({
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            })
        except Exception as e:
            tool_result_blocks.append({
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            })
    return tool_result_blocks


def get_text_edit_schema(model):
    # Current stub for Claude 4.x models. Older models use different type strings
    # (text_editor_20250124 for 3.7-sonnet, text_editor_20241022 for 3.5-sonnet).
    return {
        "type": "text_editor_20250728",
        "name": "str_replace_based_edit_tool",
    }


def run_conversation(messages):
    while True:
        response = chat(messages, tools=[get_text_edit_schema(model)])

        add_assistant_message(messages, response)
        print(text_from_message(response))

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

    return messages

## Demo 1 — summarize a file

Claude issues a `view` command against `main.py`, reads it, and summarizes.

In [5]:
messages = []
add_user_message(messages, "Open the ./main.py file and summarize its contents")

run_conversation(messages)


## Summary of ./main.py

The file contains a simple Python script with the following components:

1. **`greet(name)` function** (lines 1-2): Takes a name parameter and returns a greeting string using an f-string format (`"Hello, {name}!"`).

2. **`add(a, b)` function** (lines 5-6): Takes two parameters and returns their sum.

3. **Main execution block** (lines 9-11): Uses the `if __name__ == "__main__":` idiom to execute code only when the script is run directly:
   - Calls `greet("world")` and prints the result
   - Calls `add(2, 3)` and prints the result (which would be 5)

This is a basic introductory Python script demonstrating function definitions and the main execution pattern. When run, it would output:
```
Hello, world!
5
```


[{'role': 'user',
  'content': 'Open the ./main.py file and summarize its contents'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01Lm58FJy4n62e3THUv1MkhP', caller=DirectCaller(type='direct'), input={'command': 'view', 'path': './main.py'}, name='str_replace_based_edit_tool', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01Lm58FJy4n62e3THUv1MkhP',
    'content': '"1: def greet(name):\\n2:     return f\\"Hello, {name}!\\"\\n3: \\n4: \\n5: def add(a, b):\\n6:     return a + b\\n7: \\n8: \\n9: if __name__ == \\"__main__\\":\\n10:     print(greet(\\"world\\"))\\n11:     print(add(2, 3))\\n12: "',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='## Summary of ./main.py\n\nThe file contains a simple Python script with the following components:\n\n1. **`greet(name)` function** (lines 1-2): Takes a name parameter and returns a greeting string using an f-string format (`"Hello, 

## Demo 2 — edit a file and create another

Now Claude edits `main.py` (via `str_replace`/`insert`) and `create`s a `test.py`. Watch it `view` first, then modify. Afterward, check the sandbox folder — real files were written.

In [6]:
messages = []
add_user_message(
    messages,
    "Open the ./main.py file and add a function to calculate pi to the 5th digit. "
    "Then create a ./test.py file to test your implementation.",
)

run_conversation(messages)

I'll help you with that. Let me first view the current main.py file, then add the function and create the test file.
Let me try viewing the main.py file directly:
Great! Now I'll add a function to calculate pi to the 5th digit and create a test file. I'll use the Bailey–Borwein–Plouffe formula or Machin's formula to calculate pi.
Now let me create the test.py file:
Perfect! Let me verify the updated main.py file:
Excellent! I've successfully completed both tasks:

## Summary

### 1. **Added `calculate_pi_5_digits()` function to main.py**
   - Uses **Machin's formula**: π/4 = 4·arctan(1/5) - arctan(1/239)
   - Implements arctan using Taylor series expansion
   - Uses Python's `Decimal` module for high precision arithmetic
   - Returns π rounded to 5 decimal places: **3.14159**

### 2. **Created test.py with comprehensive tests**
   - Tests the existing `greet()` and `add()` functions
   - Tests the new `calculate_pi_5_digits()` function
   - Verifies:
     - Return type is float
     - 

[{'role': 'user',
  'content': 'Open the ./main.py file and add a function to calculate pi to the 5th digit. Then create a ./test.py file to test your implementation.'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll help you with that. Let me first view the current main.py file, then add the function and create the test file.", type='text'),
   ToolUseBlock(id='toolu_012Y5WLhTtuUQo3iEN76NwZm', caller=DirectCaller(type='direct'), input={'command': 'view', 'path': '/'}, name='str_replace_based_edit_tool', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_012Y5WLhTtuUQo3iEN76NwZm',
    'content': "Error: Access denied: Path '/' is outside the allowed directory",
    'is_error': True}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='Let me try viewing the main.py file directly:', type='text'),
   ToolUseBlock(id='toolu_016yQaaBAafA7pVQMXMJAWaj', caller=DirectCaller(type='direct'), input={

In [7]:
# Inspect what Claude wrote
for name in sorted(os.listdir(SANDBOX)):
    if name.endswith(".py"):
        print(f"===== {name} =====")
        with open(os.path.join(SANDBOX, name), encoding="utf-8") as f:
            print(f.read())

===== main.py =====
def greet(name):
    return f"Hello, {name}!"


def add(a, b):
    return a + b


def calculate_pi_5_digits():
    """
    Calculate pi to the 5th decimal digit using the Machin formula.
    Machin's formula: pi/4 = 4*arctan(1/5) - arctan(1/239)
    Returns pi rounded to 5 decimal places: 3.14159
    """
    from decimal import Decimal, getcontext
    
    # Set precision high enough to calculate pi to 5 decimal places
    getcontext().prec = 50
    
    def arctan(x, num_terms=100):
        """Calculate arctan(x) using Taylor series"""
        x = Decimal(x)
        power = x
        result = power
        for n in range(1, num_terms):
            power *= -x * x
            result += power / (2 * n + 1)
        return result
    
    # Machin's formula: pi/4 = 4*arctan(1/5) - arctan(1/239)
    pi = 4 * (4 * arctan(Decimal(1) / Decimal(5)) - arctan(Decimal(1) / Decimal(239)))
    
    # Round to 5 decimal places
    return round(float(pi), 5)


if __name__ == "__ma

## Why this matters

The text editor tool lets you embed a capable, AI-driven file editor **inside your own app** — for programmatic file edits, headless environments, or building your own coding-assistant UX with fine-grained control over how Claude touches the filesystem. You own the implementation, so you decide the sandbox, the safety checks, and what "edit a file" actually does.

Next: **The web search tool** — a fully server-side tool (Anthropic runs it; no implementation needed on your side).